**Risk Modelling - Static risk Modelling - Part 1**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

DATA_DIR = '../data'

customers = pd.read_csv(f'{DATA_DIR}/customers.csv')
accounts = pd.read_csv(f'{DATA_DIR}/accounts.csv')
transactions = pd.read_csv(f'{DATA_DIR}/transactions.csv', parse_dates=['timestamp'])
ground_truth = pd.read_csv(f'{DATA_DIR}/ground_truth.csv')
merchants = pd.read_csv(f'{DATA_DIR}/merchants.csv')
employers = pd.read_csv(f'{DATA_DIR}/employers.csv')
counterparties = pd.read_csv(f'{DATA_DIR}/counterparties.csv')

Every customer comes with a risk, the important thing is for the financial institutions to decide if a customer is worth the risk or not. For this they use 2 types of risk assesment, the first one is static and the second one is dynamic. I'll discuss the former one here now and part two of this notebook will be part 2 for dynamic risk modelling. I'm excited. 

So our dataset has many columns, some of the columns are not relavent for this particular modeling, I'm going to explore the different columns and it's potential weigtage towards static model. 

While I created this static risk model, I thought, I might join the customer table with accounts table to find how many account each customer holds, with my assumption being, high number of accounts = higher risk. But I chose not to do this because the weitage I would give to the newly created number of accounts would be quite low it doesn't make lot of sense to include it, especially because I couln't find a clear corelation between high number of account and higher AML risk. 

I'll join the customer table with employee table to get the industry the customer works in. This I think would be quite useful as someone works in goverment has lower risk than someone who works in precious metals or Casinos. 

In [2]:
# merging with employer table
risk = customers.copy()
risk = risk.merge(
    employers[['employer_id', 'industry']],
    on='employer_id',
    how='left'
)


risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education


In [3]:
risk.columns

Index(['customer_id', 'segment', 'name', 'income_or_turnover', 'budget_pct',
       'expected_monthly_outflow', 'expected_monthly_txns', 'salary_day',
       'employer_id', 'fav_merchants', 'fav_counterparties', 'kyc_risk',
       'join_date', 'industry'],
      dtype='str')

In [4]:
risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education


In [5]:
kyc_scores = {'Low': 0.10, 'Medium': 0.50, 'High': 0.85}
risk['kyc_score'] = risk['kyc_risk'].map(kyc_scores).fillna(1.00)

segment_scores = {'retail': 0.20, 'sme': 0.55, 'corporate': 0.80}
risk['segment_score'] = risk['segment'].map(segment_scores)

industry_scores = {
    'Education': 0.15,
    'Tech': 0.20,
    'Healthcare': 0.30,
    'Finance': 0.55,
    'Retail': 0.65,
    'Construction': 0.80
}
risk['industry_score'] = risk['industry'].map(industry_scores)

def financial_score(x):
    if pd.isna(x): return 0.50
    if x < 0.50: return 0.15
    if x < 0.65: return 0.25
    if x < 0.80: return 0.40
    if x <= 0.90: return 0.60
    return 0.80

risk['financial_score'] = risk['budget_pct'].apply(financial_score)

def activity_score(row):
    txns, segment = row['expected_monthly_txns'], row['segment']
    if pd.isna(txns): return 0.50

    if segment == 'retail':
        if txns <= 15: return 0.15
        if txns <= 30: return 0.25
        if txns <= 45: return 0.40
        return 0.55
    else:
        if txns <= 60: return 0.30
        if txns <= 90: return 0.45
        if txns <= 120: return 0.60
        return 0.75

risk['activity_score'] = risk.apply(activity_score, axis=1)

risk['join_date'] = pd.to_datetime(risk['join_date'])
risk['relationship_age_months'] = (
    (pd.Timestamp('2023-01-01') - risk['join_date']).dt.days / 30.44
)

def relationship_score(months):
    if pd.isna(months): return 0.50
    if months < 3: return 0.80
    if months < 6: return 0.65
    if months < 12: return 0.50
    if months < 24: return 0.35
    if months < 36: return 0.25
    return 0.15

risk['relationship_age_score'] = risk['relationship_age_months'].apply(relationship_score)

weights = {
    'kyc_score': 0.35,
    'industry_score': 0.20,
    'segment_score': 0.10,
    'financial_score': 0.15,
    'activity_score': 0.05,
    'relationship_age_score': 0.10
}

def calculate_static_risk(row):
    weighted_sum = sum(
        row[col] * weight
        for col, weight in weights.items()
        if pd.notna(row[col])
    )
    applicable_weight = sum(
        weight for col, weight in weights.items()
        if pd.notna(row[col])
    )
    return weighted_sum / applicable_weight if applicable_weight else np.nan

risk['static_risk_score'] = risk.apply(calculate_static_risk, axis=1)

In [6]:
risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry,kyc_score,segment_score,industry_score,financial_score,activity_score,relationship_age_months,relationship_age_score,static_risk_score
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance,0.1,0.20,0.55,0.60,0.15,3.022339,0.65,0.344737
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech,0.5,0.20,0.20,0.40,0.25,46.977661,0.15,0.339474
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN,0.1,0.55,NaN,0.25,0.75,31.011827,0.25,0.253333
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN,0.1,0.80,NaN,0.60,0.60,13.009198,0.35,0.360000
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education,0.1,0.20,0.15,0.40,0.40,3.022339,0.65,0.242105


I'm merging the tables so I can get the customer's employment industry and use it as one of the static risk factors. The industry can indicate the inherent AML risk associated with certain types of businesses. For example, construction and retail may have higher risk scores due to factors such as cash intensity and complex payment flows, while education or tech are assigned lower scores in this model.

Note that I have skipped the industry SMEs and corporates work in. This is because I somehow missed adding this when I generated the data. I could have added it now, but changing the underlying data at this point is not ideal and will probably make my life a bit more difficult than what it already is now. Sorry about that. Instead, for SME and corporate customers where industry data is unavailable, I simply exclude the industry factor from their calculation and normalise the remaining weights so their final static risk score still stays between 0 and 1.

At work, I obviously would not skip these details. Industry information should be collected properly for all relevant customer types during onboarding and KYC.

The weights are mainly based on how directly each factor relates to AML risk and how much useful information it gives us at the customer level. I gave **KYC the highest weight at 35%** because it is the most direct indicator of the customer's existing risk profile, and missing KYC is treated as the highest score because we don't have enough information to properly assess the customer. **Industry gets 20%** because some industries naturally have higher AML exposure, while **segment gets 10%** because corporate and SME relationships are generally more complex than retail.

The **financial profile gets 15%** and looks at how much of the customer's declared income/turnover is expected to be spent. **Expected transaction activity gets only 5%** because high transaction volume by itself is not suspicious, especially for businesses. Finally, **relationship age gets 10%** because newer customers have less established history, so there is more uncertainty around their normal behaviour.

For the individual scores, I have kept low-risk values close to **0.1–0.2**, medium-risk values around **0.4–0.6**, and higher-risk values around **0.7–0.9**. I don't use 0 or 1 for most normal cases because low risk does not mean zero risk, and high risk does not automatically mean suspicious activity. The final score is therefore a **baseline risk indicator**, which I will later combine with the dynamic transaction risk where the dynamic component will have the higher weight.


Cool, Now we have the static risk for every customer, we are going to dive into the dynamic risk rating, which ofcourse holds a higher weitage in the composit risk score, and often this is the element that moves customers from regular CDD to EDD. 

***Part 2 Dynamic Risk score***  - Dynamic risk = behavior observed after onboarding

I'm going to use one of my favorate ideas from statistics, that is standard deviation. Don't worry it basically gives you an idea of how much the current value is deviated from the average. In Dynamic scoring model, I'm gonna look at each customers and see how their transactions and activities are deviates from what I expect from them and will assign a score to this deviation, larger the deviation from expected behaviour, higher risk. On top of this I'll use some other techniques too, I will explain the rational along the way. But before that I have to create some more fancy ratios and columns that will help me create this dynamic risk model, these values are the one going to attribute to each one's dynamic score. This may be split into different parts

*Cash intensity*

More cash activity itself is not a problem, but because it is cash and cash is somewhat hard to trace, it needs to be given some more attention that what banks would usually give to someone who only do digital transaction. But how much cash transaction is too much? I don't know!! So I have created a ratio, the more cash activity you do to your total transaction, the higher the risk score will be for you. 

In [7]:
credits = transactions[transactions['direction'] == 'Credit']
cash_by_cust = credits[credits['transaction_type'] == 'Cash Deposit'].groupby('customer_id')['amount_aed_equivalent'].sum()
total_credit_by_cust = credits.groupby('customer_id')['amount_aed_equivalent'].sum()

cash_intensity = (cash_by_cust / total_credit_by_cust).fillna(0).rename('cash_intensity')
risk = risk.merge(cash_intensity, on='customer_id', how='left')
risk['cash_intensity'] = risk['cash_intensity'].fillna(0)

# stats of distribution
print(f"Total customers: {len(risk)}")
print(f"Zero values: {(risk['cash_intensity'] == 0).sum()}")
print(f"Non-zero values: {(risk['cash_intensity'] > 0).sum()}")
print(f"Zero proportion: {(risk['cash_intensity'] == 0).mean():.2%}")
print("\nSummary statistics for non-zero values:")
print(risk[risk['cash_intensity'] > 0]['cash_intensity'].describe())

Total customers: 10000
Zero values: 9851
Non-zero values: 149
Zero proportion: 98.51%

Summary statistics for non-zero values:
count    149.000000
mean       0.187943
std        0.167132
min        0.004305
25%        0.054775
50%        0.135370
75%        0.265216
max        0.680026
Name: cash_intensity, dtype: float64


Hmm, Looks like the number of cash credit transaction per customer is quite low. Cash intensity can be used as a risk factor by itself. Percentile will also be a good option

In [8]:
high_risk_countries = ['Iran', 'Syria', 'North Korea', 'Myanmar']
offshore_countries = ['Cayman Islands', 'BVI', 'Panama', 'Seychelles']

txn_with_cp = transactions.merge(
    counterparties[['cp_id', 'country']],
    left_on='counterparty_id', right_on='cp_id', how='left'
)

txn_with_cp['is_high_risk'] = txn_with_cp['country'].isin(high_risk_countries)
txn_with_cp['is_offshore'] = txn_with_cp['country'].isin(offshore_countries)

hr_exposure = txn_with_cp.groupby('customer_id')['is_high_risk'].any().rename('has_high_risk_exposure')
offshore_ratio = txn_with_cp.groupby('customer_id')['is_offshore'].mean().rename('offshore_txn_ratio')

risk = risk.merge(hr_exposure, on='customer_id', how='left').merge(offshore_ratio, on='customer_id', how='left')
risk['has_high_risk_exposure'] = risk['has_high_risk_exposure'].fillna(False)
risk['offshore_txn_ratio'] = risk['offshore_txn_ratio'].fillna(0)

In [9]:
risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry,kyc_score,segment_score,industry_score,financial_score,activity_score,relationship_age_months,relationship_age_score,static_risk_score,cash_intensity,has_high_risk_exposure,offshore_txn_ratio
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance,0.1,0.20,0.55,0.60,0.15,3.022339,0.65,0.344737,0.0,False,0.0
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech,0.5,0.20,0.20,0.40,0.25,46.977661,0.15,0.339474,0.0,False,0.0
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN,0.1,0.55,NaN,0.25,0.75,31.011827,0.25,0.253333,0.0,False,0.0
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN,0.1,0.80,NaN,0.60,0.60,13.009198,0.35,0.360000,0.0,False,0.0
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education,0.1,0.20,0.15,0.40,0.40,3.022339,0.65,0.242105,0.0,False,0.0


In [10]:
print("=== OFFSHORE TRANSACTION RATIO STATISTICS ===")
print(f"Total customers: {len(risk)}")
print(f"Zero values: {(risk['offshore_txn_ratio'] == 0).sum()}")
print(f"Non-zero values: {(risk['offshore_txn_ratio'] > 0).sum()}")
print(f"Zero proportion: {(risk['offshore_txn_ratio'] == 0).mean():.2%}")
print("\nSummary statistics for all customers:")
print(risk['offshore_txn_ratio'].describe())
print("\nSummary statistics for non-zero values:")
print(risk[risk['offshore_txn_ratio'] > 0]['offshore_txn_ratio'].describe())

=== OFFSHORE TRANSACTION RATIO STATISTICS ===
Total customers: 10000
Zero values: 9852
Non-zero values: 148
Zero proportion: 98.52%

Summary statistics for all customers:
count    10000.000000
mean         0.000035
std          0.000370
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.010989
Name: offshore_txn_ratio, dtype: float64

Summary statistics for non-zero values:
count    148.000000
mean       0.002389
std        0.001909
min        0.000355
25%        0.001114
50%        0.001953
75%        0.002815
max        0.010989
Name: offshore_txn_ratio, dtype: float64


In [11]:
risk['flag_high_risk'] = risk['has_high_risk_exposure'].astype(int)
risk['flag_cash'] = (risk['cash_intensity'] > 0).astype(int)
risk['flag_offshore'] = (risk['offshore_txn_ratio'] > 0).astype(int)

nonzero_cash = risk['cash_intensity'] > 0
risk['cash_severity'] = 0.0
risk.loc[nonzero_cash, 'cash_severity'] = (
    risk.loc[nonzero_cash, 'cash_intensity'].rank(pct=True, method='min') * 100
)

nonzero_offshore = risk['offshore_txn_ratio'] > 0
risk['offshore_severity'] = 0.0
risk.loc[nonzero_offshore, 'offshore_severity'] = (
    risk.loc[nonzero_offshore, 'offshore_txn_ratio'].rank(pct=True, method='min') * 100
)

risk['cash_component'] = risk['flag_cash'] * (
    15 + 15 * risk['cash_severity'] / 100
)

risk['offshore_component'] = risk['flag_offshore'] * (
    10 + 10 * risk['offshore_severity'] / 100
)

risk['high_risk_component'] = risk['flag_high_risk'] * 50

risk['risk_score_off_highJur'] = (
    risk['high_risk_component']
    + risk['cash_component']
    + risk['offshore_component']
).clip(0, 100)

In [12]:
risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry,kyc_score,segment_score,industry_score,financial_score,activity_score,relationship_age_months,relationship_age_score,static_risk_score,cash_intensity,has_high_risk_exposure,offshore_txn_ratio,flag_high_risk,flag_cash,flag_offshore,cash_severity,offshore_severity,cash_component,offshore_component,high_risk_component,risk_score_off_highJur
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance,0.1,0.20,0.55,0.60,0.15,3.022339,0.65,0.344737,0.0,False,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech,0.5,0.20,0.20,0.40,0.25,46.977661,0.15,0.339474,0.0,False,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN,0.1,0.55,NaN,0.25,0.75,31.011827,0.25,0.253333,0.0,False,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN,0.1,0.80,NaN,0.60,0.60,13.009198,0.35,0.360000,0.0,False,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education,0.1,0.20,0.15,0.40,0.40,3.022339,0.65,0.242105,0.0,False,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0


For this component, I'm looking at three transaction-related risk indicators: exposure to high-risk jurisdictions, cash activity and offshore transactions. High-risk jurisdiction exposure gets a fixed 50 points because I consider it a significant risk indicator on its own. For cash and offshore activity, I first check whether the customer has the behaviour at all. If they do, they receive a base score, and then I use percentile ranking among only customers who have that behaviour to measure how severe it is relative to others. This means having cash or offshore activity creates some baseline risk, while customers with more intense activity receive additional points. The final weighting is up to 50 points for high-risk jurisdictions, 30 for cash and 20 for offshore activity, giving this component a maximum score of 100.

In [13]:
cols_to_drop = ['cash_intensity_x', 'cash_intensity_y', 'cash_intensity', 'has_high_risk_exposure', 'offshore_txn_ratio', 'flag_high_risk', 'flag_cash', 'flag_offshore', 'cash_severity', 'offshore_severity', 'cash_component', 'offshore_component', 'high_risk_component', 'risk_category', 'segment_score', 'industry_score', 'financial_score', 'activity_score', 'relationship_age_months', 'relationship_age_score']
risk.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [14]:
risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry,kyc_score,static_risk_score,risk_score_off_highJur
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance,0.1,0.344737,0.0
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech,0.5,0.339474,0.0
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN,0.1,0.253333,0.0
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN,0.1,0.360000,0.0
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education,0.1,0.242105,0.0


We are moving to velocity component of our dynamic risk model

**Velocity score** - Dynamic score continuation

velocity means how quickly or heavily money is moving through a customer's account relative to their expected financial capacity.

In [15]:
# velocity_ratio = total_debit / annual_income_or_turnover
debits = transactions[transactions['direction'] == 'Debit']
total_debit_by_cust = (
    debits.groupby('customer_id')['amount_aed_equivalent']
    .sum()
    .rename('total_debit')
)

risk = risk.merge(total_debit_by_cust, on='customer_id', how='left')
risk['total_debit'] = risk['total_debit'].fillna(0)

risk['annual_income_or_turnover'] = risk['income_or_turnover'] * 12

risk['velocity_ratio'] = (
    risk['total_debit'] / risk['annual_income_or_turnover']
)

risk['velocity_ratio'].describe()

count    10000.000000
mean         1.252309
std          3.160928
min          0.001347
25%          0.458146
50%          0.800818
75%          1.146513
max        134.621122
Name: velocity_ratio, dtype: float64

In [16]:
risk.groupby('segment')['velocity_ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
segment,,,,,,,,
corporate,465.0,0.101582,0.152712,0.001358,0.018211,0.045446,0.119213,1.248924
retail,8082.0,1.524864,3.459605,0.326095,0.700316,0.903897,1.542771,134.621122
sme,1453.0,0.104540,0.172485,0.001347,0.017668,0.047780,0.117103,1.482690


The velocity ratio compares each customer's annual total debit with their annualised declared income or turnover. The results show a clear difference between segments. Retail customers have much higher ratios, with a median of 0.90, meaning their annual debit is generally close to their annual declared income. SME and corporate customers have much lower ratios, with median values of around 0.05. The retail segment also contains significant outliers, with a maximum ratio of 134.62, indicating some customers have transaction outflows far exceeding their declared income.

for example in retail, the maximum ration is 134.62, means they have that much transaction without declared income. This is highly suspecious for money laundering, and will come with further investigations including reverifying source of wealth and source of funds. 

The max retail value is skewing the average velocity score towards right.


In [17]:
risk['velocity_severity'] = (
    risk.groupby('segment')['velocity_ratio']
    .rank(pct=True, method='min') * 100
)

risk['velocity_component'] = 20 * (risk['velocity_severity'] / 100)

risk['velocity_component'].describe()
risk.groupby('segment')['velocity_component'].describe()

,count,mean,std,min,25%,50%,75%,max
segment,,,,,,,,
corporate,465.0,10.021505,5.779707,0.043011,5.032258,10.021505,15.010753,20.0
retail,8082.0,10.001235,5.773861,0.002475,5.001856,10.001237,15.000619,20.0
sme,1453.0,10.006882,5.775489,0.013765,5.010323,10.006882,15.003441,20.0


The `velocity_ratio` is very different across customer segments. Retail customers have a much higher average ratio than SME and corporate customers, mainly because their income and spending patterns are different. Because of this, I calculate the severity percentile **within each segment**, so each customer is compared only with similar customer types.

I’m giving velocity a maximum weight of **20 points**, keeping it in line with offshore exposure. Both are behavioural intensity signals, while high-risk jurisdiction exposure gets the higher 50-point weight.

Every customer has a nonzero `velocity_ratio` in this dataset, so there is no zero-dilution issue like we had with cash and offshore activity. The percentile can therefore be calculated directly within each segment.

Rolling-window velocity, such as multiple transactions within a short period, is a separate transaction-level detection method and will be handled later in Phase 3.


Now I'm going to combine these scores together to make the combined dynamic score and later combine it with static risk score to get composite risk rating. 

In [18]:
risk['dynamic_risk_raw'] = (
    0.8 * risk['risk_score_off_highJur'] +
    0.2 * (risk['velocity_component'] * 5)
)

risk['dynamic_risk_raw'].describe()

count    10000.000000
mean        10.449695
std          6.542584
min          0.002475
25%          5.089706
50%         10.191784
75%         15.312131
max         51.187110
Name: dynamic_risk_raw, dtype: float64

In [19]:
risk['static_norm'] = risk['static_risk_score'] / risk['static_risk_score'].max()
risk['dynamic_norm'] = risk['dynamic_risk_raw'] / risk['dynamic_risk_raw'].max()

risk['composite_crr'] = 0.30 * risk['static_norm'] + 0.70 * risk['dynamic_norm']

risk['composite_crr'].describe()

count    10000.000000
mean         0.261420
std          0.099088
min          0.056254
25%          0.183802
50%          0.260714
75%          0.329613
max          0.775448
Name: composite_crr, dtype: float64

Combining Dynamic Components & Composite CRR

For the dynamic score, I’m combining risk_score_off_highJur and velocity_component using an 80/20 split instead of simply adding them. This keeps the final dynamic score on a consistent 0–100 scale. I’m giving more weight to jurisdiction, cash and offshore exposure because these are stronger AML risk indicators, while velocity is more of a supporting behavioural signal.

The dynamic_risk_raw score has a mean of around 10.4, with most customers sitting at relatively low risk and a smaller group making up the higher-risk tail.

For the final CRR, I’m first normalising the static and dynamic scores to the same 0–1 scale so that one doesn’t dominate just because its raw score is larger. I’m then using 30% static and 70% dynamic, since this is a post-onboarding model where we now have actual transaction behaviour to work with. The idea is that what the customer is actually doing should carry more weight than their original profile once enough transaction history is available.

The resulting composite_crr has a mean of around 0.26 and a maximum of 0.78. I’m keeping the direct high-risk jurisdiction override separate because a customer could have an otherwise low overall score but still require immediate escalation if they have exposure to a sanctioned or otherwise high-risk jurisdiction.

***Risk drift - declared KYC tier vs behavior derived tier***

In [20]:
risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry,kyc_score,static_risk_score,risk_score_off_highJur,total_debit,annual_income_or_turnover,velocity_ratio,velocity_severity,velocity_component,dynamic_risk_raw,static_norm,dynamic_norm,composite_crr
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance,0.1,0.344737,0.0,52266.09,58152,0.898784,49.480327,9.896065,9.896065,0.420411,0.193331,0.261455
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech,0.5,0.339474,0.0,822965.88,2400000,0.342902,0.395942,0.079188,0.079188,0.413992,0.001547,0.125281
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN,0.1,0.253333,0.0,717162.13,20867868,0.034367,41.913283,8.382657,8.382657,0.308943,0.163765,0.207318
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN,0.1,0.360000,0.0,2134953.87,10719804,0.199160,85.376344,17.075269,17.075269,0.439024,0.333585,0.365217
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education,0.1,0.242105,0.0,201160.55,212016,0.948799,54.243999,10.848800,10.848800,0.295250,0.211944,0.236936


In [21]:
risk.columns

Index(['customer_id', 'segment', 'name', 'income_or_turnover', 'budget_pct',
       'expected_monthly_outflow', 'expected_monthly_txns', 'salary_day',
       'employer_id', 'fav_merchants', 'fav_counterparties', 'kyc_risk',
       'join_date', 'industry', 'kyc_score', 'static_risk_score',
       'risk_score_off_highJur', 'total_debit', 'annual_income_or_turnover',
       'velocity_ratio', 'velocity_severity', 'velocity_component',
       'dynamic_risk_raw', 'static_norm', 'dynamic_norm', 'composite_crr'],
      dtype='str')

In [23]:
risk['crr_tier'] = pd.cut(
    risk['composite_crr'],
    bins=[-0.01, 0.20, 0.40, 1.01],
    labels=['Low', 'Medium', 'High']
).astype(str)

tier_order = {'Low': 0, 'Medium': 1, 'High': 2}

risk['drift'] = risk['crr_tier'].map(tier_order) - risk['kyc_risk'].map(tier_order)

risk[risk['drift'] > 0][['customer_id', 'segment', 'kyc_risk', 'crr_tier', 'drift']].sort_values('drift', ascending=False)

,customer_id,segment,kyc_risk,crr_tier,drift
4535,CUST_104535,retail,Low,High,2.0
839,CUST_100839,corporate,Low,High,2.0
4483,CUST_104483,corporate,Low,High,2.0
2052,CUST_102052,retail,Low,High,2.0
2056,CUST_102056,retail,Low,High,2.0
...,...,...,...,...,...
9989,CUST_109989,retail,Low,Medium,1.0
9990,CUST_109990,retail,Low,Medium,1.0
216,CUST_100216,retail,Low,Medium,1.0
9994,CUST_109994,retail,Low,Medium,1.0


In [24]:
print("=== CRR Tier Distribution ===")
print(risk['crr_tier'].value_counts())
print(risk['crr_tier'].value_counts(normalize=True) * 100)

print("\n=== Declared KYC Tier Distribution ===")
print(risk['kyc_risk'].value_counts())

print("\n=== Drift Distribution ===")
print(risk['drift'].value_counts().sort_index())

print("\n=== Customers with Positive Drift (Under-rated at Onboarding) ===")
drifted = risk[risk['drift'] > 0]
print(f"Count: {len(drifted)} ({len(drifted)/len(risk)*100:.1f}% of customer base)")
print(drifted.groupby('segment').size())

print("\n=== Composite CRR by Segment ===")
print(risk.groupby('segment')['composite_crr'].describe())

print("\n=== Ground Truth Cross-Check (customers in injected scenarios) ===")
scenario_customers = (
    ground_truth.merge(accounts[['account_id', 'customer_id']], on='account_id')
    ['customer_id'].unique()
)
risk['involved_in_scenario'] = risk['customer_id'].isin(scenario_customers)
print(risk.groupby('involved_in_scenario')['composite_crr'].describe())

=== CRR Tier Distribution ===
crr_tier
Medium    6231
Low       3062
High       707
Name: count, dtype: int64
crr_tier
Medium    62.31
Low       30.62
High       7.07
Name: proportion, dtype: float64

=== Declared KYC Tier Distribution ===
kyc_risk
Low       6871
Medium    2407
High       515
Name: count, dtype: int64

=== Drift Distribution ===
drift
-2.0      21
-1.0     784
 0.0    4431
 1.0    4360
 2.0     197
Name: count, dtype: int64

=== Customers with Positive Drift (Under-rated at Onboarding) ===
Count: 4557 (45.6% of customer base)
segment
corporate     230
retail       3645
sme           682
dtype: int64

=== Composite CRR by Segment ===
            count      mean       std       min       25%       50%       75%  \
segment                                                                         
corporate   465.0  0.282932  0.094822  0.094949  0.207967  0.284561  0.353912   
retail     8082.0  0.259178  0.099607  0.056254  0.181343  0.258326  0.327018   
sme        1453.0 

**Phase 2 Results: Customer Risk Rating Summary**

**Tier distribution:** The composite CRR model classified 30.6% of customers as Low (SDD), 62.3% as Medium (CDD), and 7.1% as High (EDD). This is quite different from the original KYC rating, where 68.7% were Low, 24.1% Medium and 5.2% High. This shows that once actual transaction behaviour is included, a large portion of customers who were originally rated Low are now being moved into Medium risk. This is what I would expect from a continuously monitored risk model, since the original KYC rating is based mainly on the customer's profile at onboarding, while the CRR also considers what the customer is actually doing.

**Risk drift:** 45.6% of customers have positive drift, meaning their behaviour-based risk tier is higher than their original KYC tier. Within this group, 197 customers moved up by two full tiers, for example from Low at onboarding to High based on their current behaviour. In a real AML setup, these would be the customers I would prioritise for further review or EDD. Most of the positive drift comes from retail customers (3,645 customers), but this is also because retail makes up around 81% of the dataset. Looking at the proportion within each segment, all three segments still show substantial positive drift, so this does not appear to be something driven only by the retail segment. There are also 805 customers with negative drift, where their original KYC rating is higher than what their current behaviour suggests. This is also realistic, as some customers may have been rated conservatively during onboarding and could potentially be reviewed for a lower level of due diligence.

**Segment-level composite scores:** The average composite scores are quite similar across the three segments, ranging from around 0.26 to 0.28. I think this is a good sign because it suggests that the model isn't simply giving one segment higher scores because of differences in income or turnover. Using within-segment velocity ranking and then combining the static and dynamic scores helped keep the scoring reasonably balanced across the different customer types.

**Ground truth validation (post-hoc only):** Finally, I checked the model against the AML scenarios that were injected into the dataset, including structuring, rapid movement and layering. Customers involved in these scenarios had an average composite CRR of 0.42 compared with 0.25 for the general population, which is roughly 67% higher. There was also relatively little overlap between the two groups: the 25th percentile of scenario-involved customers (0.34) was already higher than the 75th percentile of customers not involved in the scenarios (0.32). I did not use these scenario labels when building or tuning the model. I only used them at this final stage to see whether the model was actually picking up the customers associated with the injected AML behaviour. The separation gives some confidence that the static + dynamic risk framework, which was built based on AML reasoning rather than the ground-truth labels, is actually surfacing the higher-risk population.


In [25]:
risk.to_csv('../data/customer_risk_profiles.csv', index=False)
risk.shape

(10000, 29)

In [27]:
risk.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date,industry,kyc_score,static_risk_score,risk_score_off_highJur,total_debit,annual_income_or_turnover,velocity_ratio,velocity_severity,velocity_component,dynamic_risk_raw,static_norm,dynamic_norm,composite_crr,crr_tier,drift,involved_in_scenario
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01,Finance,0.1,0.344737,0.0,52266.09,58152,0.898784,49.480327,9.896065,9.896065,0.420411,0.193331,0.261455,Medium,1.0,False
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01,Tech,0.5,0.339474,0.0,822965.88,2400000,0.342902,0.395942,0.079188,0.079188,0.413992,0.001547,0.125281,Low,-1.0,False
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01,NaN,0.1,0.253333,0.0,717162.13,20867868,0.034367,41.913283,8.382657,8.382657,0.308943,0.163765,0.207318,Medium,1.0,False
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01,NaN,0.1,0.360000,0.0,2134953.87,10719804,0.199160,85.376344,17.075269,17.075269,0.439024,0.333585,0.365217,Medium,1.0,False
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01,Education,0.1,0.242105,0.0,201160.55,212016,0.948799,54.243999,10.848800,10.848800,0.295250,0.211944,0.236936,Medium,1.0,False
